In [8]:
import numpy as np
import pandas as pd
from tqdm.autonotebook import  tqdm
from pathlib import  Path

```
\begin{table}[htbp]
    \centering
    \begin{tabular}{rrrrccr}
\toprule
Ex. & Win. & Step & Mult. & Avg. & Eng & F1 \\
\midrule
5 & 500 & 100 & 1.20 & per & Y & 0.43 \\
15 & 500 & 100 & 1.20 & per & Y & 0.41 \\
25 & 500 & 100 & 1.20 & per & Y & 0.39 \\
5 & 500 & 100 & 1.20 & per & N & 0.38 \\
25 & 500 & 100 & 1.20 & per & N & 0.37 \\
\bottomrule
\end{tabular}

    \caption{Top 5 settings for SignCLIP based Sign-Spotting across all 3 videos. Ex.= Number of examples. Win.= Window size (ms). Step= Window step size (ms). Mult.= Multiplier used to set threshold. Avg.= Whether to average score per-gloss, or across all glosses before thresholding. Eng= Whether to embed glosses as English text. F1= Micro-averaged F1 score.}
    \label{tab:signclip_signspotting_across_3_videos}
\end{table}
```

In [28]:
df = pd.read_csv("combined.csv", index=False)

TypeError: read_csv() got an unexpected keyword argument 'index'

In [12]:

# def load_csvs(base_dir: Path, pattern: str = "*heightmult_1.2/*", limit: int = 2) -> list[pd.DataFrame]:
#     """
#     Find directories matching a pattern, then load up to `limit` CSV files
#     named '*filtered.csv' inside them.
#     """
#     dfs: list[pd.DataFrame] = []
#     base_dir = base_dir.expanduser().resolve()

#     # Find all directories that match the pattern
#     dirs = [p for p in base_dir.rglob("*") if p.is_dir() and "heightmult_1.2" in str(p)]

#     print("Found %d directories", len(dirs))

#     for d in tqdm(dirs, desc="Loading CSVs"):
#         # Match CSVs inside directory
#         csvs = sorted(d.glob("*filtered.csv"))
#         for csv in csvs[:limit]:  # like `head -n2`
#             try:
#                 df = pd.read_csv(csv)
                
#                 dfs.append(df)
#             except OSError as e:
#                 print("Could not read %s: %s", csv, e)

#     return pd.concat(dfs)

In [16]:
# df = load_csvs(Path("/opt/home/cleong/projects/semantic-sign-language-search/setup_signCLIP/fairseq/examples/MMPT/results/asl_finetune_checkpoint_best/samplespergloss_5/start_0_end_None/windowsize500_step100/"))

In [30]:
df.columns

Index(['query_label', 'max_peak', 'mean_peak', 'min_peak', 'predictions_count',
       'ground_truth_count', 'true_positives', 'false_positives',
       'false_negatives', 'precision', 'recall', 'f1', 'path', 'parent_name',
       'grandparent_name', 'grandparent_path', 'English', 'samples_per_gloss',
       'windowsize_ms', 'step_ms', 'cbt', 'prominence', 'heightmult',
       'gloss_label_count'],
      dtype='object')

In [33]:
def create_latex_table(df, columns=None, count=None, out="latex_table.tex"):
  df = df.copy()

  # only include columns
  if columns is not None:
    df = df[columns]

  # rename settings columns to replace underscores with spaces
  df.columns = [col.replace("_", " ") for col in df.columns]

  # Title Case
  df.columns = [col.title() for col in df.columns]

  

  # take the top count
  if count is not None:
    df = df.head(count)


  # save to latex, only 2 decimal points for numbers
  df.to_latex(out, index=False, float_format="{:.2f}".format)

In [47]:
filtered_df = df[df['gloss_label_count'] == 1].copy()
filtered_df = filtered_df[filtered_df['query_label'] != "TOTAL"]
filtered_df = filtered_df[filtered_df['heightmult'] == 1.2]
filtered_df = filtered_df[filtered_df['English']]
filtered_df = filtered_df[filtered_df['windowsize_ms'] == 500]
filtered_df = filtered_df[filtered_df['samples_per_gloss'] == 5]
filtered_df = filtered_df.sort_values(by='f1', ascending=False)
filtered_df[['query_label','predictions_count',
       'ground_truth_count', 'true_positives', 'false_positives',
       'false_negatives', 'precision', 'recall', 'f1']]


# Aggregate counts for each gloss
agg_df = (
    filtered_df.groupby("query_label", as_index=False)
    .agg(
        predictions_count=("predictions_count", "sum"),
        ground_truth_count=("ground_truth_count", "sum"),
        true_positives=("true_positives", "sum"),
        false_positives=("false_positives", "sum"),
        false_negatives=("false_negatives", "sum"),
    )
)

# Recalculate precision, recall, and f1
agg_df["precision"] = agg_df["true_positives"] / (
    agg_df["true_positives"] + agg_df["false_positives"]
)
agg_df["recall"] = agg_df["true_positives"] / (
    agg_df["true_positives"] + agg_df["false_negatives"]
)
agg_df["f1"] = 2 * (
    agg_df["precision"] * agg_df["recall"]
) / (agg_df["precision"] + agg_df["recall"])

# Handle NaNs if denominators are 0
agg_df = agg_df.fillna(0)

# Sort for convenience
agg_df = agg_df.sort_values(by="f1", ascending=False)

# Preview
print(agg_df.head())

# Now create LaTeX table with the aggregated results
create_latex_table(
    agg_df,
    out="results_analysis/top_setting/top_setting_performance_by_gloss.tex"
)

create_latex_table(
    agg_df,
    columns=["query_label", "precision", "recall", "f1"],
    out="results_analysis/top_setting/top_setting_performance_by_gloss_abbreviated.tex"
)

   query_label  predictions_count  ground_truth_count  true_positives  \
4          DAY                 12                  12              11   
14      PEOPLE                 11                  19              11   
9       HEAVEN                 15                   8               9   
6        ENEMY                  4                   2               2   
13         MAN                 16                  20              12   

    false_positives  false_negatives  precision    recall        f1  
4                 1                2   0.916667  0.846154  0.880000  
14                0                8   1.000000  0.578947  0.733333  
9                 6                2   0.600000  0.818182  0.692308  
6                 2                0   0.500000  1.000000  0.666667  
13                4                9   0.750000  0.571429  0.648649  


In [37]:
create_latex_table(filtered_df, columns=['query_label','precision','recall','f1'], out="results_analysis/top_setting/top_setting_performance_by_gloss_abbreviated.tex")

In [38]:
filtered_df = df[df['gloss_label_count'] == 1]
filtered_df = filtered_df[filtered_df['query_label'] != "TOTAL"]
# filtered_df = filtered_df[filtered_df['query_label'] == "GOD"]
filtered_df = filtered_df[filtered_df['heightmult'] == 1.1]
filtered_df = filtered_df[filtered_df['English']]
filtered_df = filtered_df[filtered_df['windowsize_ms'] == 500]
filtered_df = filtered_df[filtered_df['samples_per_gloss'] == 5]
filtered_df = filtered_df.sort_values(by='f1', ascending=False)
filtered_df[['query_label','predictions_count',
       'ground_truth_count', 'true_positives', 'false_positives',
       'false_negatives', 'precision', 'recall', 'f1']]

,query_label,predictions_count,ground_truth_count,true_positives,false_positives,false_negatives,precision,recall,f1
1576,GOD,64,24,21,43,6,0.3281,0.7778,0.4615
1678,GOD,74,3,2,72,1,0.0270,0.6667,0.0519
